# Rail-V vibration / acoustic training

Runs the pipeline in `Rail-V/vibration` on the data you downloaded to Google Drive.

**Before starting:** Runtime > Change runtime type > **T4 GPU** (needed for step 6 only).

Steps: mount Drive, get the code, inspect the data, build the manifest and splits, run the random-forest baseline, fine-tune AST.

## 1. Mount Drive and set paths
Change `DATA` if you downloaded somewhere else. Results go to `RUNS` on Drive so they survive the session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/rail-v_dataset'   # folder download.py --out wrote into
RUNS = '/content/drive/MyDrive/rail-v-runs'
import os; os.makedirs(RUNS, exist_ok=True)

## 2. Get the code
Use `main` once the pipeline PR is merged. If the repo is private, put a GitHub token in `TOKEN` (Colab: key icon > add secret `GITHUB_TOKEN`).

In [ ]:
BRANCH = 'main'
TOKEN = ''
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
auth = f'{TOKEN}@' if TOKEN else ''
!rm -rf /content/Research-Projects
!git clone -q -b $BRANCH https://{auth}github.com/Zcodeod-OG/Research-Projects.git /content/Research-Projects
%cd /content/Research-Projects/Rail-V/vibration
!pip -q install -r requirements.txt

## 3. Inspect the data
Copy this cell's output into the project thread. It shows the folder layout and the UPM corrugation `.mat` variables, which the corrugation loader still needs.

In [ ]:
!python inspect_data.py $DATA

## 3b. Cut the corrugation runs into labelled clips
Once only. Labels come from the two corrugated sections reported in the dataset's paper (km 4.855-4.914 and 5.239-5.326). Writes 1 s WAV clips next to the `.mat` files.

In [ ]:
!python prepare_corrugation.py --data $DATA

## 3c. Cut Rail-VIVID into labelled clips
Once only, before step 4. Clips of 20 m of track around the paper's 9 known anomalies are labelled `joint`, clips well away from every stretch the detector found are `normal`. Channels 1 and 3, split by run (repeat 4 of each speed = test).

In [ ]:
!python prepare_vivid.py --data $DATA

## 4. Build the manifest and grouped splits
Writes one row per recording and fixes train / val / test once. Check the class table it prints: each class should have roughly 240 clips.

In [ ]:
!python build_manifest.py --data $DATA --out $RUNS/manifest.csv

## 5. Baseline: MFCC + random forest
CPU only, a few minutes. This is the number the deep model has to beat.

In [ ]:
!python baseline_rf.py --manifest $RUNS/manifest.csv --out $RUNS/baseline_rf --cv

## 6. Fine-tune AST (GPU)
Downloads the AudioSet-pretrained AST, fine-tunes on 5 s windows and keeps the best checkpoint on Drive. Expect a few minutes per epoch on a T4; loading the clips from Drive takes a few minutes first.

In [ ]:
!python train_ast.py --manifest $RUNS/manifest.csv --out $RUNS/ast

## 6b. Corrugation: corrugated vs normal track
A separate two-class task per sensor, trained on the 40/50 km/h runs, validated on 60 km/h and tested on 75 km/h. The axlebox accelerometer is usually the stronger signal for corrugation.

In [ ]:
!python baseline_rf.py --manifest $RUNS/manifest.csv --source corrugation_axle --fs 4000 --cv --out $RUNS/corr_axle_rf
!python baseline_rf.py --manifest $RUNS/manifest.csv --source corrugation_mic --cv --out $RUNS/corr_mic_rf
!python train_ast.py --manifest $RUNS/manifest.csv --source corrugation_mic --window 1 --hop 0.5 --out $RUNS/corr_mic_ast

## 6c. Track-anomaly detector (Rail-VIVID, unsupervised)
Finds places that look unusual in most of the 20 runs. Reads the 20 large CSVs straight from Drive (large files are fine; it's thousands of small files that break the mount). Takes several minutes.

It is scored against the paper's 9 known anomalies (Table 2, in `vivid_anomalies.csv`): a known anomaly counts as found when a flagged site is within 15 m of it.

In [ ]:
!python anomaly_vivid.py --data /content/drive/MyDrive/rail-v_dataset --out $RUNS/vivid_anomaly --anomalies vivid_anomalies.csv
from IPython.display import Image, display
display(Image(f'{RUNS}/vivid_anomaly/anomaly_profile.png'))

## 6d. Diagnose the anomaly detector
Checks timing and GPS, how far apart the two travel directions sit, and whether the known anomalies show up as impacts. Saves `impact_heatmap.png`: visible joints are vertical stripes on the blue lines.

In [ ]:
!python diagnose_vivid.py --data /content/drive/MyDrive/rail-v_dataset --out $RUNS/vivid_diag
from IPython.display import Image, display
display(Image(f'{RUNS}/vivid_diag/impact_heatmap.png'))

## 6e. Which channel and band sees the track?
For every channel and frequency band: how well it repeats between runs, the GPS shift per run (from the slow signal of the curves), and whether it rises at the 9 known anomalies (p < 0.05). Plots the best one.

In [ ]:
!python sweep_vivid.py --data /content/drive/MyDrive/rail-v_dataset --out $RUNS/vivid_sweep
from IPython.display import Image, display
display(Image(f'{RUNS}/vivid_sweep/sweep_best.png'))

## 6f. Track-feature detector (v2)
Finds the stretches of track whose vibration differs from the rest in every run, and the steps at their ends, using only the features that repeat between runs. Scores them against the 9 known anomalies. Stretches (with GPS) go to `stretches.csv`, step sites to `candidates.csv`.

In [ ]:
!python detect_vivid.py --data /content/drive/MyDrive/rail-v_dataset --out $RUNS/vivid_detect
from IPython.display import Image, display
display(Image(f'{RUNS}/vivid_detect/detect.png'))

## 7. Compare
Clip-level macro-F1 on the same held-out test split.

In [ ]:
import json
for name in ['baseline_rf', 'ast', 'corr_axle_rf', 'corr_mic_rf', 'corr_mic_ast']:
    p = f'{RUNS}/{name}/results.json'
    if os.path.exists(p):
        t = json.load(open(p))['test']['clip']
        print(f"{name:12} clip acc {t['accuracy']}  macro-F1 {t['macro_f1']}  recall {t['recall']}")

## 8. One model for every fault (vibration + sound)
A single AST encoder shared by all datasets, one head per fault family (rail surface, corrugation, joints). Each source is scored on its own held-out split, so compare with the single-task numbers above; the RF line gives the Rail-VIVID single-task reference. GPU, longer than step 6.

In [ ]:
!python baseline_rf.py --manifest $RUNS/manifest.csv --source vivid_joint --fs 2000 --window 1 --hop 0.5 --out $RUNS/vivid_rf
!python train_multitask.py --manifest $RUNS/manifest.csv --cache $RUNS/multitask_signals.pkl --out $RUNS/multitask

## 9. One predictor for every fault
Four runs of the shared model never matched all the separate models at once, so each fault keeps its best model and `predict.py` runs them together. This re-runs the two forests so they save `model.joblib`, then checks every model on its test split (expect about 1.00 / 0.94 / 0.90 / 0.79). Steps 6 and 6b must have been run, so `$RUNS/ast` and `$RUNS/corr_mic_ast` exist.

In [ ]:
!python baseline_rf.py --manifest $RUNS/manifest.csv --source corrugation_axle --fs 4000 --out $RUNS/corr_axle_rf
!python baseline_rf.py --manifest $RUNS/manifest.csv --source vivid_joint --fs 2000 --window 1 --hop 0.5 --out $RUNS/vivid_rf
!python predict.py --runs $RUNS --manifest $RUNS/manifest.csv --out $RUNS/predictor_check.json